In [3]:
from pathlib import Path
import sys

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (
            (candidate / "requirements.txt").is_file()
            and (candidate / "libs").is_dir()
        ):
            return candidate

    raise RuntimeError("Project root could not be found.")

ROOT = find_project_root()

DATA_DIR = ROOT / "data"
RESULTS_DIR = ROOT / "results"

REPORTS_DIR = RESULTS_DIR / "reports"
CURVES_DIR = RESULTS_DIR / "curves_data"
AGGREGATED_REPORTS_DIR = RESULTS_DIR / "aggregated_reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
CURVES_DIR.mkdir(parents=True, exist_ok=True)
AGGREGATED_REPORTS_DIR.mkdir(parents=True, exist_ok=True)


In [4]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Updated: Path to the data_botiot folder and specific files (1 to 4)
caminho_arquivos = DATA_DIR / "botiot"
arquivos = [os.path.join(caminho_arquivos, f"reduced_data_{x}.csv") for x in range(1, 5)]

print(f"Reading {len(arquivos)} CSV files from folder '{caminho_arquivos}'...")

# low_memory=False avoids mixed-type issues during reading
df_list = [pd.read_csv(f, low_memory=False) for f in arquivos]
df_full = pd.concat(df_list, ignore_index=True)

print("Adjusting labels...")
df_full.rename(columns={'attack': 'label', 'category': 'attack_cat'}, inplace=True)

# 2. TARGETED REMOVAL: Prevents One-Hot Encoding explosion
colunas_para_remover = [
    'pkSeqID', 'subcategory', 'saddr', 'daddr', 
    'sport', 'dport', # <-- Responsible for high RAM usage
    'state_number', 'proto_number', 'flgs_number' # Redundant
]
df_full.drop(columns=colunas_para_remover, inplace=True, errors='ignore')

# 3. SIZE REDUCTION (Optional, but RECOMMENDED FOR WISARD)
# print("Sampling 10% of the dataset to optimize RAM usage...")
# df_full = df_full.sample(frac=0.10, random_state=42)

# 4. SPLIT AND EXPORT
print("Splitting into training and test sets...")
df_train, df_test = train_test_split(
    df_full, 
    test_size=0.30, 
    random_state=42, 
    stratify=df_full['attack_cat']
)

# Saves to the 'data' folder, where the original notebook expects to find the files
os.makedirs(caminho_arquivos, exist_ok=True)
df_train.to_csv(caminho_arquivos / "BotIoT_training-set.csv", index=False)
df_test.to_csv(caminho_arquivos / "BotIoT_testing-set.csv", index=False)

print("\nDone! Dataset cleaned and ready.")
print(f"Training Shape: {df_train.shape}")
print(f"Test Shape: {df_test.shape}")

Reading 4 CSV files from folder 'C:\Users\Lucas\Desktop\Trabalho Mestrado\data\botiot'...
Adjusting labels...
Splitting into training and test sets...

Done! Dataset cleaned and ready.
Training Shape: (2567965, 37)
Test Shape: (1100557, 37)
